# exp048: TopN N=1 PP Ablation on exp040 blend

**Base**: exp040 (LB 0.949、現状 best blend) を完全コピー
**唯一の変更**: FCS spec を paper 256 流に変更
- 旧 (exp040): `FCS_TOP_K = 2, FCS_POWER = 0.4` (top-2 平均 × 0.4 乗)
- 新 (exp048): `FCS_TOP_K = 1, FCS_POWER = 1.0` (max prob × 直接乗算 = paper 256 TopN N=1)

**期待 LB**: 0.949 → 0.950-0.954 (+0.001-0.005、gold border 突破可能性)
**根拠**: paper 256 BC25 2位 stepwise ablation で TopN N=1 PP = +0.014 LB 寄与、未試行

---

# exp040 = NB4 v11 + ResSSM (0.30) + Tucker SED (0.40) + exp029 (0.30)

## 構成
- **Slot 1**: NB4 v11 (ProtoSSM + MLPHead + multi-seed=5 + KD + Retrieval + E19 + FCS + Season prior + Mirror) **+ ResidualSSM (second-pass)**
- **Slot 2**: Tucker SED 5-fold ONNX ensemble
- **Slot 3**: exp029 R3 eca_nfnet_l1 distill (single fold)

## exp040 (新規 vs exp038)
- NB4 stream に **ResidualSSM 追加** (BiSSMBlock 既存 class 流用、in-NB train ~20-30 sec)
- Inference: `l_blend += 0.30 * rs_model(emb, l_blend)` (logit-space correction)
- 既存 PP / multi-seed / KD は全 keep

## 期待 LB
- exp038 base 0.949 → **0.950-0.951** (NB4 v11 standalone 0.926 + ResSSM +0.001-0.002 を blend で transfer)


In [ ]:
import subprocess, sys, os, time

START = time.time()

# Find perch-onnx dataset
ONNX_DS = None
for _c in [
    "/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026",
    "/kaggle/input/perch-onnx-for-birdclef-2026",
    "/kaggle/input/perch-onnx-for-birdclef2026",
]:
    if os.path.isdir(_c):
        ONNX_DS = _c
        break

if ONNX_DS is None:
    print("Available /kaggle/input/:")
    for d in sorted(os.listdir("/kaggle/input/")):
        print(f"  {d}")
        sub = os.path.join("/kaggle/input", d)
        if os.path.isdir(sub):
            for f in sorted(os.listdir(sub))[:5]:
                print(f"    {f}")
    raise FileNotFoundError("perch-onnx dataset not found")

print(f"ONNX dataset: {ONNX_DS}")
print(f"Files: {os.listdir(ONNX_DS)}")

# Install onnxruntime from dataset wheel
whls = [f for f in os.listdir(ONNX_DS) if f.endswith(".whl")]
if whls:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           os.path.join(ONNX_DS, whls[0])])
    print(f"Installed: {whls[0]}")
else:
    print("No wheel found, using pre-installed onnxruntime")

In [ ]:
import gc, re, warnings, glob, random
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import onnxruntime as ort

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim.swa_utils import AveragedModel

warnings.filterwarnings("ignore")
DEVICE = "cpu"

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything(SEED)

print(f"onnxruntime {ort.__version__}, torch {torch.__version__}, seed={SEED}")

In [ ]:
# CONFIG (NB3 v29 + NB2 v10 統合)
SR = 32_000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
N_WINDOWS = 12

BASE = Path("/kaggle/input/competitions/birdclef-2026")
if not BASE.exists():
    BASE = Path("/kaggle/input/birdclef-2026")

EMB_DIR = Path("/kaggle/input/notebooks/maekeso/birdclef2026-exp010-nb1-embedding")
ANURA_DIR = Path("/kaggle/input/datasets/maekeso/birdclef2026-perch-embed-anura")
INAT_DIR  = Path("/kaggle/input/datasets/maekeso/birdclef2026-perch-embed-inat-nonbird")
TEST_DIR = BASE / "test_soundscapes"
TRAIN_SC_DIR = BASE / "train_soundscapes"
TAXONOMY_CSV = BASE / "taxonomy.csv"
SC_LABELS_CSV = BASE / "train_soundscapes_labels.csv"

# ONNX_DS is set in install cell
LABELS_CSV = os.path.join(ONNX_DS, "labels.csv")
ONNX_MODEL = os.path.join(ONNX_DS, "perch_v2.onnx")

# ProtoSSM config (NB3 v29)
D_MODEL = 128
D_STATE = 16
N_SSM_LAYERS = 2

# MLP Head config (NB2 v10)
MLP_HIDDEN = 256

# Common training hyperparams
DROPOUT = 0.1
N_EPOCHS = 80
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 15
BATCH_ONNX = 48

# Metadata embedding (site/hour parsed from filename)
N_SITES = 32
META_DIM = 8

# Multi-seed ensemble
SEEDS = [42, 123, 777, 2024, 9999]

# v4 = v2 base (SWA on) + Knowledge Distillation.
# v3 (Focal γ=2.0 単独) LB 0.920 → revert. Focal は pos_weight と過剰圧力で悪化。
# SWA は v2 で ±0.000 だったが v2 を baseline と決めたので維持。
USE_SWA = True
SWA_START_FRAC = 0.65

# Knowledge Distillation (v4): Perch の sigmoid output を soft target として
# loss = BCE(out, hard) + LAMBDA_KD * BCE(out, sigmoid(perch_logit))
# 66 files の partial label noise を Perch baseline で正則化する目的。
LAMBDA_KD = 0.15

# Aggregation across (TTA x seeds)
AGG_MODE = "mean"
BLEND_ALPHA = 0.5

# TTA shifts
TTA_SHIFTS = [-1, 0, 1]

# Prior tables (site/hour co-occurrence)
LAMBDA_PRIOR = 0.3
PRIOR_STRENGTH_SITE = 8.0
PRIOR_STRENGTH_HOUR = 8.0
PRIOR_STRENGTH_SH = 4.0

# Hierarchical Site-Conditioned KNN retrieval (inference-only)
RETRIEVAL_K = 10
RETRIEVAL_TAU = 0.05
RETRIEVAL_ALPHA_SITE = 1.5
RETRIEVAL_ALPHA_HOUR = 1.2
LAMBDA_RETRIEVAL = 0.10
RETRIEVAL_EPS = 1e-4

# BLEND weights (logit space, sum=1.0 not required but kept symmetric for first try)
W_PROTO = 0.5
W_MLP = 0.5

# file_confidence_scale
# ★ exp048: paper 256 TopN N=1 spec (was FCS_TOP_K=2, FCS_POWER=0.4 in exp040)
FCS_TOP_K = 1
FCS_POWER = 1.0

# v11: E19 — file-level species consistency boost (logit space)
# EDA: 隣接 window Jaccard 0.918 = 同一ファイル内 species 構成は静的
# boosted = (1 - BETA) * window_logit + BETA * file_signal
USE_E19 = True
E19_AGG  = "max"   # "max" / "mean" / "median"
E19_BETA = 0.2

# Train audio retrieval pool (v6): expand soundscape pool (792) → TA pool (265k)
USE_TA_RETRIEVAL = True
RETRIEVAL_TA_K = 20
# v10: class-specific LAMBDA — Aves keeps 0.05, non-Aves boosted 3x to 0.15
# (external non-Aves pool {AnuraSet, iNat} added in v9 had ±0.000 effect with uniform 0.05)
LAMBDA_RETRIEVAL_TA = 0.05         # legacy scalar (kept for fallback / Aves)
LAMBDA_TA_AVES     = 0.05
LAMBDA_TA_NONAVES  = 0.15

META_PAT = re.compile(r"_S(\d{2})_(\d{8})_(\d{2})\d{4}")

print(f"BASE: {BASE}")
print(f"EMB_DIR: {EMB_DIR}")
print(f"Files: {sorted(os.listdir(EMB_DIR))}")

In [ ]:
# TAXONOMY & PERCH LABEL MAPPING
taxonomy = pd.read_csv(TAXONOMY_CSV)
PRIMARY_LABELS = sorted(taxonomy["primary_label"].tolist())
N_CLASSES = len(PRIMARY_LABELS)
label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}

bc_labels = (
    pd.read_csv(LABELS_CSV)
    .reset_index()
    .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
)
NO_LABEL_INDEX = len(bc_labels)

taxonomy_m = taxonomy.copy()
taxonomy_m["scientific_name_lookup"] = taxonomy_m["scientific_name"]
bc_lookup = bc_labels.rename(columns={"scientific_name": "scientific_name_lookup"})

mapping = taxonomy_m.merge(
    bc_lookup[["scientific_name_lookup", "bc_index"]],
    on="scientific_name_lookup", how="left",
)
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL_INDEX).astype(int)
label_to_bc = mapping.set_index("primary_label")["bc_index"]

BC_INDICES = np.array([int(label_to_bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC = BC_INDICES[MAPPED_MASK].astype(np.int32)

proxy_map = {}
unmapped_df = mapping[mapping["bc_index"] == NO_LABEL_INDEX].copy()
unmapped_non_sono = unmapped_df[
    ~unmapped_df["primary_label"].astype(str).str.contains("son", na=False)
]
for _, row in unmapped_non_sono.iterrows():
    genus = str(row["scientific_name"]).split()[0]
    hits = bc_labels[
        bc_labels["scientific_name"].astype(str).str.match(
            rf"^{re.escape(genus)}\s", na=False
        )
    ]
    if len(hits) > 0:
        proxy_map[label_to_idx[row["primary_label"]]] = (
            hits["bc_index"].astype(int).values
        )

print(f"Species: {N_CLASSES}, Mapped: {MAPPED_MASK.sum()}, Proxies: {len(proxy_map)}")

# v10: class-specific LAMBDA_TA vector — non-Aves species get 3x boost
class_name_arr = np.array([
    taxonomy.set_index("primary_label").loc[lbl, "class_name"]
    for lbl in PRIMARY_LABELS
])
NON_AVES_MASK = (class_name_arr != "Aves")
LAMBDA_TA_VEC = np.full(N_CLASSES, LAMBDA_TA_AVES, dtype=np.float32)
LAMBDA_TA_VEC[NON_AVES_MASK] = LAMBDA_TA_NONAVES
print(f"LAMBDA_TA: Aves={LAMBDA_TA_AVES} ({(~NON_AVES_MASK).sum()} sp), "
      f"non-Aves={LAMBDA_TA_NONAVES} ({NON_AVES_MASK.sum()} sp)")

In [ ]:
# LOAD SOUNDSCAPE EMBEDDINGS + LABELS
sc_data = np.load(EMB_DIR / "soundscape_embeddings.npz")
sc_emb = sc_data["embeddings"].astype(np.float32)
sc_scores = sc_data["scores"].astype(np.float32)
sc_meta = pd.read_parquet(EMB_DIR / "soundscape_meta.parquet")

print(f"Soundscapes: {sc_emb.shape[0]} windows, {sc_meta['filename'].nunique()} files")

sc_labels_df = pd.read_csv(SC_LABELS_CSV)
labeled_files = set(sc_labels_df["filename"].unique())
print(f"Labeled files: {len(labeled_files)}")

label_map = {}
for _, r in sc_labels_df.iterrows():
    fn = r["filename"]
    end_sec = int(pd.Timedelta(r["end"]).total_seconds())
    row_id = f"{Path(fn).stem}_{end_sec}"
    labels_str = str(r["primary_label"]).split(";")
    y = np.zeros(N_CLASSES, dtype=np.float32)
    for lbl in labels_str:
        lbl = lbl.strip()
        if lbl in label_to_idx:
            y[label_to_idx[lbl]] = 1.0
    label_map[row_id] = y

is_labeled = sc_meta["filename"].isin(labeled_files).values

def reshape_to_files(arr, meta):
    fnames = meta["filename"].values
    unique = list(dict.fromkeys(fnames))
    n_files = len(unique)
    D = arr.shape[1]
    out = np.zeros((n_files, N_WINDOWS, D), dtype=arr.dtype)
    file_to_idx = {f: i for i, f in enumerate(unique)}
    counters = np.zeros(n_files, dtype=int)
    for ri, fn in enumerate(fnames):
        fi = file_to_idx[fn]
        wi = counters[fi]
        if wi < N_WINDOWS:
            out[fi, wi] = arr[ri]
            counters[fi] += 1
    return out, unique

lab_meta = sc_meta[is_labeled].reset_index(drop=True)
lab_emb_flat = sc_emb[is_labeled]
lab_scores_flat = sc_scores[is_labeled]

lab_emb_files, lab_file_list = reshape_to_files(lab_emb_flat, lab_meta)
lab_scores_files, _ = reshape_to_files(lab_scores_flat, lab_meta)

def parse_meta(fname):
    m = META_PAT.search(fname)
    if m is None:
        return 0, 0
    return int(m.group(1)), int(m.group(3))

lab_site_ids = np.array([parse_meta(fn)[0] for fn in lab_file_list], dtype=np.int64)
lab_hours = np.array([parse_meta(fn)[1] for fn in lab_file_list], dtype=np.int64)
print(f"Labeled metadata: unique sites={np.unique(lab_site_ids).tolist()}, unique hours={np.unique(lab_hours).tolist()}")

lab_labels_files = np.zeros((len(lab_file_list), N_WINDOWS, N_CLASSES), dtype=np.float32)
for fi, fn in enumerate(lab_file_list):
    stem = Path(fn).stem
    for wi in range(N_WINDOWS):
        end_sec = (wi + 1) * WINDOW_SEC
        rid = f"{stem}_{end_sec}"
        if rid in label_map:
            lab_labels_files[fi, wi] = label_map[rid]

print(f"Labeled: {lab_emb_files.shape[0]} files, {lab_emb_files.shape}")
print(f"Active classes: {int((lab_labels_files.sum(axis=(0,1)) > 0).sum())}")

# PRIOR TABLES
file_labels = (lab_labels_files.sum(axis=1) > 0).astype(np.float32)
global_p = file_labels.mean(axis=0).astype(np.float32)

prior_site_ids = sorted(set(int(s) for s in lab_site_ids))
site_to_pi = {s: i for i, s in enumerate(prior_site_ids)}
site_n = np.zeros(len(prior_site_ids), dtype=np.float32)
site_p = np.zeros((len(prior_site_ids), N_CLASSES), dtype=np.float32)
for s in prior_site_ids:
    m = (lab_site_ids == s)
    site_n[site_to_pi[s]] = m.sum()
    site_p[site_to_pi[s]] = file_labels[m].mean(axis=0)

prior_hours = sorted(set(int(h) for h in lab_hours))
hour_to_pi = {h: i for i, h in enumerate(prior_hours)}
hour_n = np.zeros(len(prior_hours), dtype=np.float32)
hour_p = np.zeros((len(prior_hours), N_CLASSES), dtype=np.float32)
for h in prior_hours:
    m = (lab_hours == h)
    hour_n[hour_to_pi[h]] = m.sum()
    hour_p[hour_to_pi[h]] = file_labels[m].mean(axis=0)

sh_to_pi = {}
sh_n_list, sh_p_list = [], []
for s in prior_site_ids:
    for h in prior_hours:
        m = (lab_site_ids == s) & (lab_hours == h)
        if m.sum() > 0:
            sh_to_pi[(s, h)] = len(sh_n_list)
            sh_n_list.append(float(m.sum()))
            sh_p_list.append(file_labels[m].mean(axis=0))
sh_n = np.array(sh_n_list, dtype=np.float32) if sh_n_list else np.zeros(0, dtype=np.float32)
sh_p = np.stack(sh_p_list).astype(np.float32) if sh_p_list else np.zeros((0, N_CLASSES), dtype=np.float32)

def compute_prior_logit(site_id, hour, eps=1e-4):
    p = global_p.astype(np.float32).copy()
    h_i = hour_to_pi.get(int(hour), -1)
    if h_i >= 0:
        nh = hour_n[h_i]
        wh = nh / (nh + PRIOR_STRENGTH_HOUR)
        p = wh * hour_p[h_i] + (1 - wh) * p
    s_i = site_to_pi.get(int(site_id), -1)
    if s_i >= 0:
        ns = site_n[s_i]
        ws = ns / (ns + PRIOR_STRENGTH_SITE)
        p = ws * site_p[s_i] + (1 - ws) * p
    sh_i = sh_to_pi.get((int(site_id), int(hour)), -1)
    if sh_i >= 0:
        nsh = sh_n[sh_i]
        wsh = nsh / (nsh + PRIOR_STRENGTH_SH)
        p = wsh * sh_p[sh_i] + (1 - wsh) * p
    p = np.clip(p, eps, 1 - eps)
    return (np.log(p) - np.log1p(-p)).astype(np.float32)

lab_prior_files = np.stack(
    [compute_prior_logit(s, h) for s, h in zip(lab_site_ids, lab_hours)]
).astype(np.float32)
print(f"Prior tables: sites={len(prior_site_ids)}, hours={len(prior_hours)}, "
      f"sh={len(sh_to_pi)}; lab_prior shape={lab_prior_files.shape}")

# RETRIEVAL POOL
lab_pool_emb = lab_emb_flat.astype(np.float32)
_lab_pool_norm = np.linalg.norm(lab_pool_emb, axis=1, keepdims=True) + 1e-8
lab_pool_emb_norm = (lab_pool_emb / _lab_pool_norm).astype(np.float32)

lab_pool_labels = np.zeros((len(lab_meta), N_CLASSES), dtype=np.float32)
for _i, _rid in enumerate(lab_meta["row_id"].values):
    if _rid in label_map:
        lab_pool_labels[_i] = label_map[_rid]

lab_pool_site_ids = np.zeros(len(lab_meta), dtype=np.int64)
lab_pool_hours = np.zeros(len(lab_meta), dtype=np.int64)
for _i, _fn in enumerate(lab_meta["filename"].values):
    _s, _h = parse_meta(_fn)
    lab_pool_site_ids[_i] = _s
    lab_pool_hours[_i] = _h

print(f"Retrieval pool: {lab_pool_emb_norm.shape[0]} windows, "
      f"{int((lab_pool_labels.sum(axis=0) > 0).sum())} active classes, "
      f"sites={len(set(lab_pool_site_ids.tolist()))}, hours={len(set(lab_pool_hours.tolist()))}")

# TRAIN AUDIO RETRIEVAL POOL (v6): expand from 792 SC windows to ~265k TA windows
# Focal recordings provide cleaner per-class signal than partial SC labels.
_ta_npz = EMB_DIR / "trainaudio_embeddings.npz"
_ta_pq  = EMB_DIR / "trainaudio_meta.parquet"

if USE_TA_RETRIEVAL and _ta_npz.exists() and _ta_pq.exists():
    _ta_data = np.load(_ta_npz)
    _ta_meta = pd.read_parquet(_ta_pq)

    _ta_emb_raw = _ta_data["embeddings"].astype(np.float32)
    del _ta_data
    gc.collect()

    ta_pool_labels = np.zeros((len(_ta_meta), N_CLASSES), dtype=np.float32)
    for _i, _lbl in enumerate(_ta_meta["primary_label"].values):
        if _lbl in label_to_idx:
            ta_pool_labels[_i, label_to_idx[_lbl]] = 1.0

    _ta_norm = np.linalg.norm(_ta_emb_raw, axis=1, keepdims=True) + 1e-8
    ta_pool_emb_norm = (_ta_emb_raw / _ta_norm).astype(np.float32)
    del _ta_emb_raw, _ta_norm, _ta_meta
    gc.collect()

    print(f"TA pool (BC2026): {ta_pool_emb_norm.shape[0]} windows, "
          f"{int((ta_pool_labels.sum(0) > 0).sum())} active classes")

    # ── External non-Aves pools (v9): AnuraSet + iNat non-Aves ──
    _ext_emb_chunks = []
    _ext_lbl_chunks = []
    # AnuraSet (multi-label, primary_labels csv)
    _an_npz = ANURA_DIR / "anura_embeddings.npz"
    _an_pq  = ANURA_DIR / "anura_meta.parquet"
    if _an_npz.exists() and _an_pq.exists():
        _an_data = np.load(_an_npz)
        _an_meta = pd.read_parquet(_an_pq)
        _an_emb = _an_data["embeddings"].astype(np.float32)
        del _an_data; gc.collect()
        _an_n = _an_emb.shape[0]
        _an_lbl = np.zeros((_an_n, N_CLASSES), dtype=np.float32)
        for _i, _csv in enumerate(_an_meta["primary_labels"].values):
            for _lbl in str(_csv).split(","):
                _lbl = _lbl.strip()
                if _lbl in label_to_idx:
                    _an_lbl[_i, label_to_idx[_lbl]] = 1.0
        _an_norm = np.linalg.norm(_an_emb, axis=1, keepdims=True) + 1e-8
        _an_emb_norm = (_an_emb / _an_norm).astype(np.float32)
        _ext_emb_chunks.append(_an_emb_norm)
        _ext_lbl_chunks.append(_an_lbl)
        print(f"AnuraSet pool: {_an_n} windows, "
              f"{int((_an_lbl.sum(0) > 0).sum())} active classes")
        del _an_emb, _an_norm, _an_meta; gc.collect()
    else:
        print(f"WARN: AnuraSet pool not found at {ANURA_DIR}")

    # iNat non-Aves (single-label primary_label)
    _in_npz = INAT_DIR / "inat_nonaves_embeddings.npz"
    _in_pq  = INAT_DIR / "inat_nonaves_meta.parquet"
    if _in_npz.exists() and _in_pq.exists():
        _in_data = np.load(_in_npz)
        _in_meta = pd.read_parquet(_in_pq)
        _in_emb = _in_data["embeddings"].astype(np.float32)
        del _in_data; gc.collect()
        _in_n = _in_emb.shape[0]
        _in_lbl = np.zeros((_in_n, N_CLASSES), dtype=np.float32)
        for _i, _lbl in enumerate(_in_meta["primary_label"].values):
            if _lbl in label_to_idx:
                _in_lbl[_i, label_to_idx[_lbl]] = 1.0
        _in_norm = np.linalg.norm(_in_emb, axis=1, keepdims=True) + 1e-8
        _in_emb_norm = (_in_emb / _in_norm).astype(np.float32)
        _ext_emb_chunks.append(_in_emb_norm)
        _ext_lbl_chunks.append(_in_lbl)
        print(f"iNat non-Aves pool: {_in_n} windows, "
              f"{int((_in_lbl.sum(0) > 0).sum())} active classes")
        del _in_emb, _in_norm, _in_meta; gc.collect()
    else:
        print(f"WARN: iNat pool not found at {INAT_DIR}")

    if _ext_emb_chunks:
        _ext_emb = np.concatenate(_ext_emb_chunks, axis=0)
        _ext_lbl = np.concatenate(_ext_lbl_chunks, axis=0)
        ta_pool_emb_norm = np.concatenate([ta_pool_emb_norm, _ext_emb], axis=0)
        ta_pool_labels   = np.concatenate([ta_pool_labels, _ext_lbl], axis=0)
        del _ext_emb, _ext_lbl, _ext_emb_chunks, _ext_lbl_chunks; gc.collect()
        print(f"TA pool (after external concat): {ta_pool_emb_norm.shape[0]} windows, "
              f"{int((ta_pool_labels.sum(0) > 0).sum())} active classes")
else:
    print("WARNING: trainaudio_embeddings.npz not found -- TA retrieval disabled")
    USE_TA_RETRIEVAL = False
    ta_pool_emb_norm = None
    ta_pool_labels = None

In [ ]:
# === PROTOSSM (NB3 v29) ===
class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.proj_delta = nn.Linear(d_model, d_model)
        self.proj_B = nn.Linear(d_model, d_state)
        self.proj_C = nn.Linear(d_model, d_state)
        self.proj_D = nn.Linear(d_model, d_model)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.log_A = nn.Parameter(torch.log(A))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B_sz, L, D = x.shape
        delta = F.softplus(self.proj_delta(x))
        B = self.proj_B(x)
        C = self.proj_C(x)
        D_param = self.proj_D(x)
        A = -torch.exp(self.log_A)
        h = torch.zeros(B_sz, self.d_model, self.d_state, device=x.device)
        outputs = []
        for t in range(L):
            dt = delta[:, t].unsqueeze(-1)
            dA = torch.exp(A.unsqueeze(0) * dt)
            dB = dt * B[:, t].unsqueeze(1)
            h = h * dA + x[:, t].unsqueeze(-1) * dB
            y = (h * C[:, t].unsqueeze(1)).sum(-1) + D_param[:, t]
            outputs.append(y)
        return self.dropout(torch.stack(outputs, dim=1))


class BiSSMBlock(nn.Module):
    def __init__(self, d_model, d_state, dropout=0.1):
        super().__init__()
        self.fwd_ssm = SelectiveSSM(d_model, d_state, dropout)
        self.bwd_ssm = SelectiveSSM(d_model, d_state, dropout)
        self.proj = nn.Linear(d_model * 2, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        fwd = self.fwd_ssm(x)
        bwd = self.bwd_ssm(x.flip(1)).flip(1)
        out = self.proj(torch.cat([fwd, bwd], dim=-1))
        return self.norm(x + out)


class CrossAttnBlock(nn.Module):
    def __init__(self, d_model, num_heads=2, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            d_model, num_heads, dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        a, _ = self.attn(x, x, x, need_weights=False)
        return self.norm(x + self.dropout(a))


class ProtoSSM(nn.Module):
    def __init__(self, d_input, d_model, d_state, n_ssm_layers, n_classes,
                 n_windows, dropout=0.1, n_sites=32, meta_dim=8, n_attn_heads=2):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.ssm_layers = nn.ModuleList([
            BiSSMBlock(d_model, d_state, dropout) for _ in range(n_ssm_layers)
        ])
        self.attn_layers = nn.ModuleList([
            CrossAttnBlock(d_model, num_heads=n_attn_heads, dropout=dropout)
            for _ in range(n_ssm_layers)
        ])
        self.prototypes = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.temperature = nn.Parameter(torch.tensor(10.0))
        self.bias = nn.Parameter(torch.zeros(n_classes))
        self.alpha = nn.Parameter(torch.ones(n_classes) * 0.5)

    def forward(self, emb, logits, site_ids=None, hours=None, prior_logit=None, lambda_prior=0.0):
        x = self.input_proj(emb) + self.pos_emb[:, :emb.shape[1]]
        if site_ids is not None and hours is not None:
            s_e = self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings - 1))
            h_e = self.hour_emb(hours.clamp(0, 23))
            meta = self.meta_proj(torch.cat([s_e, h_e], dim=-1))
            x = x + meta.unsqueeze(1)
        for ssm_layer, attn_layer in zip(self.ssm_layers, self.attn_layers):
            x = ssm_layer(x)
            x = attn_layer(x)
        x_norm = F.normalize(x, dim=-1)
        p_norm = F.normalize(self.prototypes, dim=-1)
        sim = torch.einsum("btd,cd->btc", x_norm, p_norm) * self.temperature + self.bias
        alpha = torch.sigmoid(self.alpha)
        out = alpha * sim + (1 - alpha) * logits
        if prior_logit is not None and lambda_prior > 0:
            out = out + lambda_prior * prior_logit.unsqueeze(1)
        return torch.sigmoid(out)

    def init_prototypes(self, emb_flat, labels_flat):
        with torch.no_grad():
            x = self.input_proj(emb_flat)
            for ci in range(self.prototypes.shape[0]):
                mask = labels_flat[:, ci] > 0.5
                if mask.sum() > 0:
                    self.prototypes[ci] = x[mask].mean(0)


# === MLP Head (NB2 v10) ===
class MLPHead(nn.Module):
    def __init__(self, d_input, d_hidden, n_classes, dropout=0.1,
                 n_sites=32, meta_dim=8):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_hidden),
            nn.LayerNorm(d_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_hidden)
        self.mlp = nn.Sequential(
            nn.Linear(d_hidden, d_hidden),
            nn.LayerNorm(d_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, n_classes),
        )
        self.temperature = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.ones(n_classes) * 0.5)

    def forward(self, emb, logits, site_ids=None, hours=None,
                prior_logit=None, lambda_prior=0.0):
        x = self.input_proj(emb)
        if site_ids is not None and hours is not None:
            s_e = self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings - 1))
            h_e = self.hour_emb(hours.clamp(0, 23))
            meta = self.meta_proj(torch.cat([s_e, h_e], dim=-1))
            x = x + meta.unsqueeze(1)
        h = self.mlp(x) * self.temperature
        alpha = torch.sigmoid(self.alpha)
        out = alpha * h + (1 - alpha) * logits
        if prior_logit is not None and lambda_prior > 0:
            out = out + lambda_prior * prior_logit.unsqueeze(1)
        return torch.sigmoid(out)


print("ProtoSSM + MLPHead defined.")

# === exp040: ResidualSSM (mtoshidesu Cell 7j 流) - second-pass error correction ===
# Input: emb (1536d) + first_pass_logits (234d) concat → correction (234d)
# Lightweight 2nd-pass、BiSSMBlock 既存 class 流用、~439K params、in-NB train ~20-30 sec
class ResidualSSM(nn.Module):
    def __init__(self, d_input=1536, d_scores=N_CLASSES,
                 d_model=64, d_state=8, n_classes=N_CLASSES,
                 dropout=0.1, n_sites=32, meta_dim=8):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(d_input + d_scores, d_model),
            nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.pos_enc = nn.Parameter(torch.randn(1, N_WINDOWS, d_model) * 0.02)
        self.ssm_block = BiSSMBlock(d_model, d_state, dropout)
        self.norm = nn.LayerNorm(d_model)
        self.output_head = nn.Linear(d_model, n_classes)
        nn.init.zeros_(self.output_head.weight)
        nn.init.zeros_(self.output_head.bias)

    def forward(self, emb, first_pass, site_ids=None, hours=None):
        B, T, _ = emb.shape
        x = torch.cat([emb, first_pass], dim=-1)
        h = self.input_proj(x) + self.pos_enc[:, :T, :]
        if site_ids is not None and hours is not None:
            s_e = self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings - 1))
            h_e = self.hour_emb(hours.clamp(0, 23))
            meta = self.meta_proj(torch.cat([s_e, h_e], dim=-1))
            h = h + meta.unsqueeze(1)
        h = self.norm(self.ssm_block(h))
        return self.output_head(h)


RESIDUAL_CORRECTION_WEIGHT = 0.30  # mtoshidesu 0.15-0.35 推奨
print("exp040: ResidualSSM defined")


In [ ]:
# TRAIN BOTH MODELS (multi-seed ensemble)
# Shared tensors
emb_flat = torch.tensor(lab_emb_flat, dtype=torch.float32)
lab_flat = torch.zeros(len(lab_meta), N_CLASSES, dtype=torch.float32)
for i, rid in enumerate(lab_meta["row_id"]):
    if rid in label_map:
        lab_flat[i] = torch.tensor(label_map[rid])

train_emb = torch.tensor(lab_emb_files, dtype=torch.float32)
train_logits = torch.tensor(lab_scores_files, dtype=torch.float32)
train_labels = torch.tensor(lab_labels_files, dtype=torch.float32)
train_site = torch.tensor(lab_site_ids, dtype=torch.long)
train_hour = torch.tensor(lab_hours, dtype=torch.long)
train_prior = torch.tensor(lab_prior_files, dtype=torch.float32)

pos_counts = train_labels.sum(dim=(0, 1)).clamp(min=1)
neg_counts = train_labels.shape[0] * train_labels.shape[1] - pos_counts
pos_weight = (neg_counts / pos_counts).clamp(max=30.0)

# KD teacher: Perch's mapped logits → sigmoid → soft target
teacher_prob = torch.sigmoid(train_logits).clamp(1e-7, 1 - 1e-7)


def _train_one(model_cls_kwargs, model_factory, seed):
    seed_everything(seed)
    model = model_factory().to(DEVICE)
    if hasattr(model, "init_prototypes"):
        model.init_prototypes(emb_flat, lab_flat)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
    best_loss = float("inf")
    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    patience_counter = 0
    epoch_done = 0

    swa_model = AveragedModel(model) if USE_SWA else None
    swa_start = int(N_EPOCHS * SWA_START_FRAC)
    swa_n = 0

    for epoch in range(N_EPOCHS):
        model.train()
        out = model(train_emb, train_logits, site_ids=train_site, hours=train_hour,
                    prior_logit=train_prior, lambda_prior=LAMBDA_PRIOR)
        # Main BCE loss (hard labels)
        loss_main = F.binary_cross_entropy(out, train_labels, reduction="none")
        loss_main = (loss_main * pos_weight.unsqueeze(0).unsqueeze(0)).mean()
        # KD loss (soft target from Perch)
        loss_kd = F.binary_cross_entropy(out, teacher_prob, reduction="mean")
        loss = loss_main + LAMBDA_KD * loss_kd
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        epoch_done = epoch + 1
        if USE_SWA and epoch >= swa_start:
            swa_model.update_parameters(model)
            swa_n += 1
        # disable early stop while SWA is collecting
        if not USE_SWA and patience_counter >= PATIENCE:
            break

    if USE_SWA and swa_n >= 1:
        model.load_state_dict(swa_model.module.state_dict())
    else:
        model.load_state_dict(best_state)
    model.eval()
    return model, best_loss, epoch_done, swa_n


# Train ProtoSSM x SEEDS
proto_models = []
t0_all = time.time()
for si, seed in enumerate(SEEDS):
    factory = lambda: ProtoSSM(
        d_input=1536, d_model=D_MODEL, d_state=D_STATE,
        n_ssm_layers=N_SSM_LAYERS, n_classes=N_CLASSES,
        n_windows=N_WINDOWS, dropout=DROPOUT,
        n_sites=N_SITES, meta_dim=META_DIM,
    )
    if si == 0:
        _tmp = factory()
        print(f"Training ProtoSSM (x{len(SEEDS)} seeds): {sum(p.numel() for p in _tmp.parameters())} params/model")
        del _tmp
    t0 = time.time()
    m, bl, ep, sn = _train_one(None, factory, seed)
    proto_models.append(m)
    print(f"  ProtoSSM[seed {seed}] done in {time.time()-t0:.1f}s, best_loss={bl:.4f}, epochs={ep}, swa_n={sn}")
print(f"ProtoSSM ensemble: {len(proto_models)} models in {time.time()-t0_all:.1f}s")

# Train MLPHead x SEEDS
mlp_models = []
t0_all = time.time()
for si, seed in enumerate(SEEDS):
    factory = lambda: MLPHead(
        d_input=1536, d_hidden=MLP_HIDDEN, n_classes=N_CLASSES,
        dropout=DROPOUT, n_sites=N_SITES, meta_dim=META_DIM,
    )
    if si == 0:
        _tmp = factory()
        print(f"Training MLPHead (x{len(SEEDS)} seeds): {sum(p.numel() for p in _tmp.parameters())} params/model")
        del _tmp
    t0 = time.time()
    m, bl, ep, sn = _train_one(None, factory, seed)
    mlp_models.append(m)
    print(f"  MLPHead[seed {seed}] done in {time.time()-t0:.1f}s, best_loss={bl:.4f}, epochs={ep}, swa_n={sn}")
print(f"MLPHead ensemble: {len(mlp_models)} models in {time.time()-t0_all:.1f}s")

# === exp040: Train ResidualSSM on labeled SS first-pass residuals ===
print("\n[exp040] Training ResidualSSM on labeled SS...")
_t0_rs = time.time()

# Compute first-pass predictions on labeled SS (single-forward across multi-seed, no TTA)
def _first_pass_single_lab(models_list, emb_t, scores_t, site_t, hour_t, prior_t):
    probs = []
    for m in models_list:
        m.eval()
        with torch.no_grad():
            p = m(emb_t, scores_t, site_ids=site_t, hours=hour_t,
                  prior_logit=prior_t, lambda_prior=LAMBDA_PRIOR)
        probs.append(p)
    return torch.stack(probs, dim=0).mean(dim=0)

n_lab = lab_emb_files.shape[0]
fp_lab_chunks = []
for fi in range(n_lab):
    emb_t = torch.tensor(lab_emb_files[fi:fi+1], dtype=torch.float32)
    sc_t = torch.tensor(lab_scores_files[fi:fi+1], dtype=torch.float32)
    site_t = torch.tensor([lab_site_ids[fi]], dtype=torch.long)
    hour_t = torch.tensor([lab_hours[fi]], dtype=torch.long)
    prior_t = torch.tensor(lab_prior_files[fi:fi+1], dtype=torch.float32)
    p_proto = _first_pass_single_lab(proto_models, emb_t, sc_t, site_t, hour_t, prior_t)
    p_mlp = _first_pass_single_lab(mlp_models, emb_t, sc_t, site_t, hour_t, prior_t)
    p_proto_c = p_proto.clamp(min=1e-7, max=1 - 1e-7)
    p_mlp_c = p_mlp.clamp(min=1e-7, max=1 - 1e-7)
    l_proto = torch.log(p_proto_c) - torch.log1p(-p_proto_c)
    l_mlp = torch.log(p_mlp_c) - torch.log1p(-p_mlp_c)
    l_blend = W_PROTO * l_proto + W_MLP * l_mlp
    fp_lab_chunks.append(l_blend.squeeze(0).cpu().numpy())
first_pass_lab = np.stack(fp_lab_chunks).astype(np.float32)
print(f"  first_pass_lab: {first_pass_lab.shape}")

fp_prob = 1.0 / (1.0 + np.exp(-np.clip(first_pass_lab, -30, 30)))
residuals_lab = (lab_labels_files - fp_prob).astype(np.float32)
print(f"  residuals: mean={residuals_lab.mean():+.4f}  std={residuals_lab.std():.4f}  abs_mean={np.abs(residuals_lab).mean():.4f}")

rs_model = ResidualSSM(d_input=lab_emb_files.shape[-1], d_scores=N_CLASSES,
                       d_model=64, d_state=8, n_classes=N_CLASSES)
print(f"  ResidualSSM params: {sum(p.numel() for p in rs_model.parameters()):,}")

rs_emb_t = torch.tensor(lab_emb_files, dtype=torch.float32)
rs_fp_t = torch.tensor(first_pass_lab, dtype=torch.float32)
rs_res_t = torch.tensor(residuals_lab, dtype=torch.float32)
rs_site_t = torch.tensor(lab_site_ids, dtype=torch.long)
rs_hour_t = torch.tensor(lab_hours, dtype=torch.long)

rs_perm = torch.randperm(n_lab, generator=torch.Generator().manual_seed(42)).numpy()
rs_n_val = max(1, int(n_lab * 0.15))
rs_val_i = rs_perm[:rs_n_val]
rs_tr_i = rs_perm[rs_n_val:]

RS_EPOCHS = 30
RS_PATIENCE = 8
rs_opt = torch.optim.AdamW(rs_model.parameters(), lr=1e-3, weight_decay=1e-3)
rs_sched = torch.optim.lr_scheduler.OneCycleLR(
    rs_opt, max_lr=1e-3, epochs=RS_EPOCHS, steps_per_epoch=1,
    pct_start=0.1, anneal_strategy="cos"
)
best_rs_loss, best_rs_state, rs_wait = float("inf"), None, 0
for ep in range(RS_EPOCHS):
    rs_model.train()
    corr = rs_model(rs_emb_t[rs_tr_i], rs_fp_t[rs_tr_i],
                    site_ids=rs_site_t[rs_tr_i], hours=rs_hour_t[rs_tr_i])
    rs_loss = F.mse_loss(corr, rs_res_t[rs_tr_i])
    rs_opt.zero_grad(); rs_loss.backward()
    torch.nn.utils.clip_grad_norm_(rs_model.parameters(), 1.0)
    rs_opt.step(); rs_sched.step()
    rs_model.eval()
    with torch.no_grad():
        v_corr = rs_model(rs_emb_t[rs_val_i], rs_fp_t[rs_val_i],
                          site_ids=rs_site_t[rs_val_i], hours=rs_hour_t[rs_val_i])
        v_loss = F.mse_loss(v_corr, rs_res_t[rs_val_i]).item()
    if v_loss < best_rs_loss:
        best_rs_loss = v_loss
        best_rs_state = {k: v.clone() for k, v in rs_model.state_dict().items()}
        rs_wait = 0
    else:
        rs_wait += 1
    if rs_wait >= RS_PATIENCE:
        print(f"  ResidualSSM early stop ep {ep+1}")
        break
rs_model.load_state_dict(best_rs_state)
rs_model.eval()
print(f"  ResidualSSM trained: best val MSE={best_rs_loss:.6f} in {time.time()-_t0_rs:.1f}s")

with torch.no_grad():
    all_corr_chk = rs_model(rs_emb_t, rs_fp_t, site_ids=rs_site_t, hours=rs_hour_t).numpy()
print(f"  correction magnitude: mean_abs={np.abs(all_corr_chk).mean():.4f}  max_abs={np.abs(all_corr_chk).max():.4f}")


In [ ]:
# PERCH ONNX FOR TEST INFERENCE (CPU)
print(f"Loading ONNX model: {ONNX_MODEL}")
sess_opts = ort.SessionOptions()
sess_opts.intra_op_num_threads = 4
session = ort.InferenceSession(ONNX_MODEL, sess_opts, providers=["CPUExecutionProvider"])


def read_audio(path, target_samples=None):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr != SR:
        import torchaudio
        y = torch.from_numpy(y).unsqueeze(0)
        y = torchaudio.functional.resample(y, sr, SR).squeeze(0).numpy()
    if target_samples is not None:
        if len(y) < target_samples:
            y = np.pad(y, (0, target_samples - len(y)))
        else:
            y = y[:target_samples]
    return y


def map_logits_to_scores(logits):
    scores = np.zeros((logits.shape[0], N_CLASSES), dtype=np.float32)
    scores[:, MAPPED_POS] = logits[:, MAPPED_BC]
    for pos, bc_idx_arr in proxy_map.items():
        scores[:, pos] = logits[:, bc_idx_arr].max(axis=1)
    return scores


def infer_perch_cpu(windows):
    all_emb, all_scores = [], []
    for i in range(0, len(windows), BATCH_ONNX):
        batch = windows[i:i + BATCH_ONNX]
        outputs = session.run(None, {"inputs": batch})
        out_dict = {o.name: v for o, v in zip(session.get_outputs(), outputs)}
        all_emb.append(out_dict["embedding"].astype(np.float32))
        all_scores.append(map_logits_to_scores(out_dict["label"].astype(np.float32)))
    return np.concatenate(all_emb), np.concatenate(all_scores)


def compute_retrieval_logit(test_emb, site_id, hour,
                            K=RETRIEVAL_K, tau=RETRIEVAL_TAU,
                            alpha_site=RETRIEVAL_ALPHA_SITE,
                            alpha_hour=RETRIEVAL_ALPHA_HOUR,
                            eps=RETRIEVAL_EPS):
    e_norm = test_emb / (np.linalg.norm(test_emb, axis=1, keepdims=True) + 1e-8)
    sim = e_norm @ lab_pool_emb_norm.T
    w = np.ones(lab_pool_emb_norm.shape[0], dtype=np.float32)
    w[lab_pool_site_ids == site_id] *= alpha_site
    w[lab_pool_hours == hour] *= alpha_hour
    w_sim = sim * w[None, :]
    K_eff = min(K, w_sim.shape[1])
    topk_idx = np.argpartition(w_sim, -K_eff, axis=1)[:, -K_eff:]
    topk_sim = np.take_along_axis(w_sim, topk_idx, axis=1)
    topk_label = lab_pool_labels[topk_idx]
    sm = topk_sim - topk_sim.max(axis=1, keepdims=True)
    sm = np.exp(sm / max(tau, 1e-6))
    sm = sm / (sm.sum(axis=1, keepdims=True) + 1e-8)
    p = (sm[..., None] * topk_label).sum(axis=1)
    p = np.clip(p, eps, 1 - eps)
    return (np.log(p) - np.log1p(-p)).astype(np.float32)


def compute_retrieval_ta_logit(test_emb,
                               K=RETRIEVAL_TA_K, tau=RETRIEVAL_TAU,
                               eps=RETRIEVAL_EPS):
    # KNN retrieval on train_audio pool (~265k focal recordings, primary_label only)
    e_norm = test_emb / (np.linalg.norm(test_emb, axis=1, keepdims=True) + 1e-8)
    sim = e_norm @ ta_pool_emb_norm.T          # (N_windows, 265k)
    K_eff = min(K, sim.shape[1])
    topk_idx = np.argpartition(sim, -K_eff, axis=1)[:, -K_eff:]
    topk_sim = np.take_along_axis(sim, topk_idx, axis=1)
    topk_label = ta_pool_labels[topk_idx]      # (N_windows, K, N_CLASSES)
    sm = topk_sim - topk_sim.max(axis=1, keepdims=True)
    sm = np.exp(sm / max(tau, 1e-6))
    sm = sm / (sm.sum(axis=1, keepdims=True) + 1e-8)
    p = (sm[..., None] * topk_label).sum(axis=1)
    p = np.clip(p, eps, 1 - eps)
    return (np.log(p) - np.log1p(-p)).astype(np.float32)


print("Perch ONNX (CPU) + retrieval helper ready.")

In [ ]:
# === Stage A: NB4 v11 + ResSSM inference ===
test_files = sorted(glob.glob(str(TEST_DIR / "*.ogg")))
if len(test_files) == 0:
    print("No test files, using train_soundscapes as fallback")
    test_files = sorted(glob.glob(str(TRAIN_SC_DIR / "*.ogg")))[:8]
print(f"Test files: {len(test_files)}")

# Load konbu17 train_audio LinearHead weights
KONBU_W = None
KONBU_B = None
KONBU_MASK = None
for p in Path("/kaggle/input").rglob("head_weights_train_audio.npz"):
    _hw = np.load(p, allow_pickle=True)
    KONBU_W = _hw["W"].astype(np.float32)        # (234, 1536)
    KONBU_B = _hw["b"].astype(np.float32)        # (234,)
    KONBU_MASK = _hw["trained_mask"].astype(np.float32)  # (234,)
    print(f"Konbu17 head loaded: {p}")
    print(f"  W={KONBU_W.shape}, b={KONBU_B.shape}, trained={int(KONBU_MASK.sum())}/234")
    break
if KONBU_W is None:
    print("WARN: konbu17 head not attached, will skip 3rd axis")

all_row_ids = []
probs_exp010 = []
probs_konbu  = []
audio_cache = []

t0 = time.time()
for _m in proto_models + mlp_models:
    _m.eval()

from concurrent.futures import ThreadPoolExecutor

def _load_windows(fp):
    y = read_audio(fp, target_samples=SR * 60)
    return y.reshape(N_WINDOWS, WINDOW_SAMPLES), y

PREFETCH = 4
executor = ThreadPoolExecutor(max_workers=4)
pending = {}
for _i in range(min(PREFETCH, len(test_files))):
    pending[_i] = executor.submit(_load_windows, test_files[_i])


def _ensemble_one(models_list, emb_t, scores_t, site_t, hour_t, prior_t):
    ens_probs = []
    for s in TTA_SHIFTS:
        if s == 0:
            e_shift, sc_shift = emb_t, scores_t
        else:
            e_shift = torch.roll(emb_t, shifts=s, dims=1)
            sc_shift = torch.roll(scores_t, shifts=s, dims=1)
        for m in models_list:
            p = m(e_shift, sc_shift, site_ids=site_t, hours=hour_t,
                  prior_logit=prior_t, lambda_prior=LAMBDA_PRIOR)
            if s != 0:
                p = torch.roll(p, shifts=-s, dims=1)
            ens_probs.append(p)
    return torch.stack(ens_probs, dim=0).mean(dim=0)


for fi, fpath in enumerate(test_files):
    stem = Path(fpath).stem
    _ni = fi + PREFETCH
    if _ni < len(test_files):
        pending[_ni] = executor.submit(_load_windows, test_files[_ni])
    windows, raw_60s = pending.pop(fi).result()
    audio_cache.append(raw_60s)

    emb, scores = infer_perch_cpu(windows)
    _m = META_PAT.search(fpath)
    site_id = int(_m.group(1)) if _m else 0
    hour = int(_m.group(3)) if _m else 0
    site_t = torch.tensor([site_id], dtype=torch.long)
    hour_t = torch.tensor([hour], dtype=torch.long)
    prior_vec = compute_prior_logit(site_id, hour)
    prior_t = torch.tensor(prior_vec, dtype=torch.float32).unsqueeze(0)

    with torch.no_grad():
        emb_t = torch.tensor(emb, dtype=torch.float32).unsqueeze(0)
        scores_t = torch.tensor(scores, dtype=torch.float32).unsqueeze(0)
        p_proto = _ensemble_one(proto_models, emb_t, scores_t, site_t, hour_t, prior_t)
        p_mlp = _ensemble_one(mlp_models, emb_t, scores_t, site_t, hour_t, prior_t)
        p_proto_c = p_proto.clamp(min=1e-7, max=1 - 1e-7)
        p_mlp_c = p_mlp.clamp(min=1e-7, max=1 - 1e-7)
        l_proto = torch.log(p_proto_c) - torch.log1p(-p_proto_c)
        l_mlp = torch.log(p_mlp_c) - torch.log1p(-p_mlp_c)
        l_blend = W_PROTO * l_proto + W_MLP * l_mlp

        # === exp040: ResidualSSM correction (second-pass) ===
        with torch.no_grad():
            rs_corr = rs_model(emb_t, l_blend.float(),
                               site_ids=site_t, hours=hour_t)
            l_blend = l_blend + RESIDUAL_CORRECTION_WEIGHT * rs_corr

        if LAMBDA_RETRIEVAL > 0:
            ret_logit = compute_retrieval_logit(emb, site_id, hour)
            ret_logit_t = torch.tensor(ret_logit, dtype=torch.float32).unsqueeze(0)
            l_blend = l_blend + LAMBDA_RETRIEVAL * ret_logit_t

        if USE_TA_RETRIEVAL and ta_pool_emb_norm is not None:
            ta_ret_logit = compute_retrieval_ta_logit(emb)
            ta_ret_logit_t = torch.tensor(ta_ret_logit, dtype=torch.float32).unsqueeze(0)
            lam_t = torch.tensor(LAMBDA_TA_VEC, dtype=torch.float32)
            l_blend = l_blend + lam_t * ta_ret_logit_t

        # E19 (NB4 v11 で +0.002 確認): file-level species consistency boost
        if USE_E19 and E19_BETA > 0:
            if E19_AGG == "max":
                file_sig = l_blend.max(dim=1, keepdim=True).values
            elif E19_AGG == "mean":
                file_sig = l_blend.mean(dim=1, keepdim=True)
            elif E19_AGG == "median":
                file_sig = l_blend.median(dim=1, keepdim=True).values
            else:
                file_sig = None
            if file_sig is not None:
                l_blend = (1.0 - E19_BETA) * l_blend + E19_BETA * file_sig

        agg = torch.sigmoid(l_blend)
        probs_e10 = agg.squeeze(0).numpy()
    probs_exp010.append(probs_e10)

    # konbu17 LinearHead on Perch emb
    if KONBU_W is not None:
        head_logit = emb.astype(np.float32) @ KONBU_W.T + KONBU_B   # (12, 234)
        head_logit = head_logit * KONBU_MASK.reshape(1, -1)          # zero untrained
        head_prob = 1.0 / (1.0 + np.exp(-np.clip(head_logit, -50, 50))).astype(np.float32)
        probs_konbu.append(head_prob)
    else:
        probs_konbu.append(np.full((N_WINDOWS, N_CLASSES), 0.5, dtype=np.float32))

    for wi in range(N_WINDOWS):
        end_sec = (wi + 1) * WINDOW_SEC
        all_row_ids.append(f"{stem}_{end_sec}")

    if (fi + 1) % 10 == 0 or fi == len(test_files) - 1:
        elapsed = time.time() - t0
        print(f"  exp010 [{fi+1}/{len(test_files)}] {elapsed:.0f}s")

executor.shutdown()
probs_exp010 = np.stack(probs_exp010)
probs_konbu  = np.stack(probs_konbu)
print(f"exp010 done: {probs_exp010.shape}, konbu={probs_konbu.shape} in {time.time()-t0:.0f}s")

In [ ]:
# === Stage B: Tucker 5-fold SED ONNX ensemble ===
import librosa
from scipy.ndimage import gaussian_filter1d

N_MELS_SED = 256
N_FFT_SED  = 2048
HOP_SED    = 512
FMIN_SED   = 20
FMAX_SED   = 16000
TOP_DB_SED = 80


def find_sed_dir():
    hits = sorted(Path("/kaggle/input").rglob("sed_fold0.onnx"))
    assert hits, "sed_fold0.onnx not found — attach tuckerarrants/bc2026-distilled-sed-public"
    return hits[0].parent


def make_sed_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 4
    so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), sess_options=so,
                                providers=["CPUExecutionProvider"])


def audio_to_mel(chunks):
    mels = []
    for x in chunks:
        s = librosa.feature.melspectrogram(
            y=x, sr=SR, n_fft=N_FFT_SED, hop_length=HOP_SED,
            n_mels=N_MELS_SED, fmin=FMIN_SED, fmax=FMAX_SED, power=2.0,
        )
        s = librosa.power_to_db(s, top_db=TOP_DB_SED)
        s = (s - s.mean()) / (s.std() + 1e-6)
        mels.append(s)
    return np.stack(mels)[:, None].astype(np.float32)


def sigmoid_sed(x):
    return (1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))).astype(np.float32)


sed_dir = find_sed_dir()
sed_fold_paths = sorted(sed_dir.glob("sed_fold*.onnx"),
                        key=lambda p: int(re.search(r"sed_fold(\d+)", p.name).group(1)))
sed_sessions = [make_sed_session(p) for p in sed_fold_paths]
print(f"SED dir: {sed_dir}")
print(f"SED folds loaded: {[p.name for p in sed_fold_paths]}")

t0 = time.time()
probs_sed = []   # (N_files, 12, 234)
for fi, raw_60s in enumerate(audio_cache):
    chunks = raw_60s.reshape(N_WINDOWS, WINDOW_SAMPLES)
    mel = audio_to_mel(chunks)
    p_sum = np.zeros((N_WINDOWS, N_CLASSES), dtype=np.float32)
    for sess in sed_sessions:
        outs = sess.run(None, {sess.get_inputs()[0].name: mel})
        clip_logits = outs[0]
        frame_max = outs[1].max(axis=1)
        p_sum += 0.5 * sigmoid_sed(clip_logits) + 0.5 * sigmoid_sed(frame_max)
    p_mean = p_sum / len(sed_sessions)
    p_mean = gaussian_filter1d(p_mean, sigma=0.65, axis=0, mode="nearest").astype(np.float32)
    probs_sed.append(p_mean)
    if (fi + 1) % 10 == 0 or fi == len(audio_cache) - 1:
        elapsed = time.time() - t0
        print(f"  SED [{fi+1}/{len(audio_cache)}] {elapsed:.0f}s")

probs_sed = np.stack(probs_sed)
print(f"SED done: {probs_sed.shape} in {time.time()-t0:.0f}s")

In [ ]:
# === exp029 R3 (eca_nfnet_l1 + Perch distill) inference ===
# Use audio_cache built by NB4 stage (60s raw per file).
# Output probs_e17 of shape (N_files, 12, N_CLASSES) in sigmoid space, gaussian smoothed.
import timm
import torchaudio

# ---- Locate exp017 R2 ckpt (Dataset 経由) ----
E17_STATE_DIR = None
for _p in [
    Path("/kaggle/input/datasets/maekeso/birdclef2026-exp029-l1-single"),
    Path("/kaggle/input/birdclef2026-exp029-l1-single"),
]:
    if _p.exists() and (any(_p.rglob("r3_fold0_ckpt_best_ns22.pth"))
                        or any(_p.rglob("r2_ckpt_best_ns22.pth"))
                        or any(_p.rglob("ckpt_best_ns22.pth"))
                        or any(_p.rglob("ckpt_latest*.pth"))):
        E17_STATE_DIR = _p; break
if E17_STATE_DIR is None:
    for _hit in Path("/kaggle/input").rglob("r3_fold0_ckpt_best_ns22.pth"):
        E17_STATE_DIR = _hit.parent; break
if E17_STATE_DIR is None:
    for _hit in Path("/kaggle/input").rglob("r2_ckpt_best_ns22.pth"):
        E17_STATE_DIR = _hit.parent; break
if E17_STATE_DIR is None:
    for _hit in Path("/kaggle/input").rglob("ckpt_best_ns22.pth"):
        E17_STATE_DIR = _hit.parent; break
assert E17_STATE_DIR is not None, "exp040: exp029 ckpt not found (looked for r3_fold0_ckpt_best_ns22.pth)"
print(f"exp040 (exp029 ckpt) state dir: {E17_STATE_DIR}")

E17_CKPT = None
for _name in [
    "r3_fold0_ckpt_best_ns22.pth",   # exp029 R3 fold0 (val 0.9409, LB 0.923)
    "r3_fold0_ckpt_best_macro.pth",
    "r3_fold0_ckpt_latest.pth",
    "ckpt_best_ns22.pth",      # R1 v2 fallback (val 0.9166)
    "ckpt_best_macro.pth",
    "ckpt_latest.pth",
]:
    _hits = list(E17_STATE_DIR.rglob(_name))
    if _hits:
        E17_CKPT = _hits[0]; break
assert E17_CKPT is not None, f"No ckpt under {E17_STATE_DIR}"
print(f"  ckpt: {E17_CKPT.name} ({E17_CKPT.stat().st_size/1e6:.1f}MB)")

# ---- exp017 config (must match training) ----
E17_BACKBONE = "eca_nfnet_l1"  # exp029 student (l0 から +30% params)
E17_N_MELS   = 256
E17_N_FFT    = 2048
E17_HOP      = 512
E17_FMIN     = 20
E17_FMAX     = 16000
E17_TRAIN_SAMPLES = SR * 5
E17_USE_DISTILL = True
E17_PERCH_DIM = 1536

# ---- Architecture (copy of exp017 BirdSEDModel) ----
class _E17MelTF(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR, n_fft=E17_N_FFT, hop_length=E17_HOP,
            n_mels=E17_N_MELS, f_min=E17_FMIN, f_max=E17_FMAX, power=2.0,
        )
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80)
    def forward(self, x):
        return self.db(self.mel_spec(x))


class _E17GeMFreq(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(float(p_init)))
        self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)


class _E17DistillHead(nn.Module):
    def __init__(self, backbone_dim, embed_dim=1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)
    def forward(self, feature_map):
        return self.proj(feature_map.mean(dim=[2, 3]))


class _E17SED(nn.Module):
    def __init__(self, backbone_name=E17_BACKBONE, num_classes=N_CLASSES,
                 drop_path_rate=0.1, hidden_dim=512):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=False, in_chans=1,
            num_classes=0, global_pool="", drop_path_rate=drop_path_rate,
        )
        with torch.no_grad():
            n_tf = E17_TRAIN_SAMPLES // E17_HOP + 1
            dummy = torch.randn(1, 1, E17_N_MELS, n_tf)
            feat = self.backbone(dummy)
            self.backbone_dim = feat.shape[1]
        self.gem_freq = _E17GeMFreq(p_init=3.0)
        self.dense = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(self.backbone_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )
        self.att = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        if E17_USE_DISTILL:
            self.distill_head = _E17DistillHead(self.backbone_dim, E17_PERCH_DIM)
    def forward(self, x, return_framewise=False):
        h = self.backbone(x)
        h_cls = h.detach() if E17_USE_DISTILL else h
        h_cls = self.gem_freq(h_cls)
        h_cls = h_cls.permute(0, 2, 1)
        h_cls = self.dense(h_cls)
        h_cls = h_cls.permute(0, 2, 1)
        norm_att = torch.softmax(torch.tanh(self.att(h_cls)), dim=-1)
        framewise_logits = self.cla(h_cls)
        clip_logits = torch.sum(norm_att * framewise_logits, dim=2)
        if return_framewise:
            return clip_logits, framewise_logits.permute(0, 2, 1)
        return clip_logits


# ---- Load weights ----
try:
    _state = torch.load(str(E17_CKPT), map_location="cpu", weights_only=False)
except TypeError:
    _state = torch.load(str(E17_CKPT), map_location="cpu")
print(f"  ckpt epoch={_state.get('epoch')}, best_ns22={_state.get('best_ns22', float('nan')):.4f}")

e17_model = _E17SED().to(torch.device("cpu"))
e17_model.load_state_dict(_state["model_state"], strict=False)
e17_model.eval()
e17_mel_tf = _E17MelTF().to(torch.device("cpu"))
print(f"  e17 loaded: {sum(p.numel() for p in e17_model.parameters())/1e6:.1f}M params")

# ---- Inference using audio_cache (60s raw per file) ----
t0 = time.time()
probs_e17 = []  # (N_files, 12, N_CLASSES)
with torch.no_grad():
    for _fi, _raw_60s in enumerate(audio_cache):
        _chunks = _raw_60s.reshape(N_WINDOWS, WINDOW_SAMPLES).astype(np.float32)
        _wav_t = torch.from_numpy(_chunks).unsqueeze(1)        # (12, 1, 160000)
        _mel = e17_mel_tf(_wav_t)
        # per-instance standardize (same as exp017 training/infer)
        for _i in range(_mel.size(0)):
            _mel[_i] = (_mel[_i] - _mel[_i].mean()) / (_mel[_i].std() + 1e-6)
        _clip, _frame = e17_model(_mel, return_framewise=True)
        _frame_max = _frame.max(dim=1).values
        # sigmoid space blend (mirrors Tucker recipe: 0.5*sigmoid(clip) + 0.5*sigmoid(frame_max))
        _p_clip   = torch.sigmoid(_clip).numpy().astype(np.float32)
        _p_frame  = torch.sigmoid(_frame_max).numpy().astype(np.float32)
        _p_mean   = 0.5 * _p_clip + 0.5 * _p_frame              # (12, N_CLASSES)
        # gaussian smooth across windows (Tucker sigma=0.65)
        _p_smooth = gaussian_filter1d(_p_mean, sigma=0.65, axis=0, mode="nearest").astype(np.float32)
        probs_e17.append(_p_smooth)
        if (_fi + 1) % 10 == 0 or _fi == len(audio_cache) - 1:
            print(f"  e17 [{_fi+1}/{len(audio_cache)}] {time.time()-t0:.0f}s")

probs_e17 = np.stack(probs_e17).astype(np.float32)
print(f"e17 done: {probs_e17.shape} in {time.time()-t0:.0f}s")


In [ ]:
# === Stage C: 3-way rank blend (NB4 + Tucker + exp029) + Sonotype mirror + submission ===
# exp040 ratio config (w30-40-30 inherited from exp038):
BLEND_W_E10  = 0.30   # exp040 (inherited from exp038): NB4 0.35→0.30
BLEND_W_SED  = 0.40   # Tucker public SED weight
BLEND_W_E17  = 0.30   # exp040 (inherited from exp038): exp029 0.25→0.30 — note var name kept "E17" for code reuse
USE_SED_PRE_AVG = False

assert abs(BLEND_W_E10 + BLEND_W_SED + BLEND_W_E17 - 1.0) < 1e-6, \
    "blend weights must sum to 1.0"

flat_e10 = probs_exp010.reshape(-1, probs_exp010.shape[-1])
flat_sed = probs_sed.reshape(-1, probs_sed.shape[-1])
flat_e17 = probs_e17.reshape(-1, probs_e17.shape[-1])  # ← actually exp029

if USE_SED_PRE_AVG:
    _eps = 1e-7
    _l_sed = np.log(np.clip(flat_sed, _eps, 1 - _eps)) - np.log1p(-np.clip(flat_sed, _eps, 1 - _eps))
    _l_e17 = np.log(np.clip(flat_e17, _eps, 1 - _eps)) - np.log1p(-np.clip(flat_e17, _eps, 1 - _eps))
    _w_sed = BLEND_W_SED / (BLEND_W_SED + BLEND_W_E17)
    _w_e17 = BLEND_W_E17 / (BLEND_W_SED + BLEND_W_E17)
    _l_sed_avg = _w_sed * _l_sed + _w_e17 * _l_e17
    flat_sed_combined = (1.0 / (1.0 + np.exp(-np.clip(_l_sed_avg, -50, 50)))).astype(np.float32)
    rank_e10 = pd.DataFrame(flat_e10).rank(axis=0, pct=True).to_numpy().astype(np.float32)
    rank_sed = pd.DataFrame(flat_sed_combined).rank(axis=0, pct=True).to_numpy().astype(np.float32)
    _w_nb4 = BLEND_W_E10
    _w_sed_total = BLEND_W_SED + BLEND_W_E17
    blend_flat = _w_nb4 * rank_e10 + _w_sed_total * rank_sed
    print(f"  SED-pre-avg mode: NB4={_w_nb4:.2f} / (Tucker+e29)={_w_sed_total:.2f} "
          f"(Tucker:e29 internal = {_w_sed:.2f}:{_w_e17:.2f})")
else:
    rank_e10 = pd.DataFrame(flat_e10).rank(axis=0, pct=True).to_numpy().astype(np.float32)
    rank_sed = pd.DataFrame(flat_sed).rank(axis=0, pct=True).to_numpy().astype(np.float32)
    rank_e17 = pd.DataFrame(flat_e17).rank(axis=0, pct=True).to_numpy().astype(np.float32)
    blend_flat = (BLEND_W_E10 * rank_e10
                  + BLEND_W_SED * rank_sed
                  + BLEND_W_E17 * rank_e17)
    print(f"  3-way rank blend: NB4={BLEND_W_E10} / Tucker={BLEND_W_SED} / e29={BLEND_W_E17}")

# === Sonotype mirror (公開 0.946 NB Cell 40) ===
MIRROR_PAIRS = (
    ("47158son15", "47158son16"),
    ("47158son09", "47158son12"),
    ("47158son02", "47158son14"),
    ("47158son13", "47158son21", "47158son22", "47158son23"),
)
col_to_idx = {lbl: i for i, lbl in enumerate(PRIMARY_LABELS)}
mirror_count = 0
for group in MIRROR_PAIRS:
    valid_idx = [col_to_idx[s] for s in group if s in col_to_idx]
    if len(valid_idx) >= 2:
        group_max = blend_flat[:, valid_idx].max(axis=1, keepdims=True)
        blend_flat[:, valid_idx] = group_max
        mirror_count += len(valid_idx)
print(f"  Sonotype mirror applied to {mirror_count} columns")

probs_blend = blend_flat.reshape(probs_exp010.shape)
print(f"blend shape: {probs_blend.shape}, mean={probs_blend.mean():.4f}, max={probs_blend.max():.4f}")

preds_array = probs_blend.reshape(-1, N_CLASSES)
_n, _c = preds_array.shape
_view = preds_array.reshape(-1, N_WINDOWS, _c)
_sorted = np.sort(_view, axis=1)
_topk_mean = _sorted[:, -FCS_TOP_K:, :].mean(axis=1, keepdims=True)
_scale = np.power(_topk_mean, FCS_POWER)
preds_array = (_view * _scale).reshape(_n, _c).astype(np.float32)

submission = pd.DataFrame(preds_array, columns=PRIMARY_LABELS)
submission.insert(0, "row_id", all_row_ids)

sample_sub = pd.read_csv(BASE / "sample_submission.csv")
expected_ids = set(sample_sub["row_id"])
our_ids = set(submission["row_id"])
missing = expected_ids - our_ids
if missing:
    print(f"WARNING: {len(missing)} missing row_ids - filling zeros")
    missing_df = pd.DataFrame({"row_id": list(missing)})
    for sp in PRIMARY_LABELS:
        missing_df[sp] = 0.0
    submission = pd.concat([submission, missing_df], ignore_index=True)
extra = our_ids - expected_ids
if extra:
    submission = submission[submission["row_id"].isin(expected_ids)]
submission = submission.set_index("row_id").loc[sample_sub["row_id"]].reset_index()
submission.to_csv("submission.csv", index=False)

total = time.time() - START
print(f"\nSubmission: {submission.shape}, total {total:.0f}s ({total/60:.1f} min)")
print(f"Mean pred: {submission[PRIMARY_LABELS].values.mean():.6f}")
print(f"Max pred:  {submission[PRIMARY_LABELS].values.max():.6f}")
print(submission.head())
